# DAOWOD — Contribution A · Master Colab Notebook

**Distribution-aware active annotation for Open-World Object Detection.**
This is the authoritative Colab entrypoint for every currently implemented and
scientifically defensible Contribution A experiment. It supersedes
`notebooks/contribution_a_colab.ipynb`.

The scientific source of truth is `docs/proposal.docx`, read into
`docs/research_design.md`. This notebook does not restate the science; it runs it.

---

## 0. What this notebook does and does not claim

### What it is

An **offline active-annotation simulation over real PROB proposals**. For a fixed
*region-level* oracle budget, it asks which candidate regions should be labelled so
that the most rare unknown objects are discovered. The score under test is the
proposal's equation (1):

$$s(x) \;=\; U(x) \;+\; \lambda \cdot D(x) \;+\; \gamma \cdot w(\hat c(x)) \cdot \mathrm{coh}(x)
\qquad w(c) \propto 1/n_c$$

| Term | Meaning |
|---|---|
| $U(x)$ | **uncertainty** — posterior entropy over the exported class distribution |
| $D(x)$ | **diversity / representativeness** — novelty against a disjoint reference bank |
| $w(\hat c(x))$ | **estimated rarity weight** — inverse frequency of the *estimated* pseudo-class |
| $\mathrm{coh}(x)$ | **local coherence** — cluster-coherence / local density in feature space |
| $w(\hat c(x)) \cdot \mathrm{coh}(x)$ | the **gated product**: rarity fires only when the candidate is rare **AND** locally supported |

$\hat c(x)$ and $\hat n_c$ are estimates, **iteratively updated after every annotation
round**, exactly as the proposal specifies. Evaluation is on the **controlled
head / medium / tail protocol**, and the x-axis is **oracle cost**, giving the
proposal's **annotation-efficiency curves**.

### What it computes — valid here

Offline **discovery** metrics over the annotation set: unknown / head / medium / tail
discovery recall, unique classes discovered, annotation precision, background and
isolated-outlier selection rates, redundancy, embedding diversity, and discovery AUC
against oracle budget, with per-seed dispersion.

### What it cannot compute — and does not claim

> **No detector is trained or evaluated in this notebook.** It therefore makes **no
> claim** about:
>
> * `known mAP` improvement
> * official `U-Recall` improvement
> * `Wilderness Impact (WI)` improvement
> * `A-OSE` improvement
> * catastrophic-forgetting reduction
> * Contribution B validation
>
> Those quantities require PROB retraining and the official evaluator. Wherever they
> would appear, the notebook writes **`NOT AVAILABLE — requires
> retraining/evaluation`**. It never substitutes a zero, a NaN or a placeholder.

Contribution B is **out of scope here**: only its mathematical allocation core exists
(`src/daowod/memory.py`), and its notebook is `contribution_b_colab.ipynb`.

### Where the science already stands

Read [`docs/results.md`](../docs/results.md) **before** interpreting any output. On real
S-OWODB Task-1 proposals the coherence gate's premise is **falsified**: background is the
most locally homogeneous stratum in the pool, and a one-line `objectness × box scale`
prior finds ~1.88× more unknown objects than the full distribution-aware score. That is a
result about a hypothesis the proposal states — not a bug to be fixed by re-running.
This notebook is built to reproduce that finding honestly, including the possibility
that it reproduces again.

### Design rules this notebook obeys

1. **Thin orchestration only.** Every scientific computation happens in `daowod` or in
   `experiments/`. No cell reimplements scoring, candidate semantics, oracle matching,
   long-tail construction, metrics or plotting.
2. **The protocol lives in `configs/contribution_a.yaml`.** Sizes, budgets, rounds,
   seeds, arms and severities come from version control, not from a notebook cell.
3. **Fail fast, never shrink.** If a mode exceeds a declared runtime/disk/RAM limit the
   run stops and names the parameter to change. It never silently reduces samples,
   seeds, strategies or severities.
4. **Restartable.** Every expensive unit is cached and resumable; an interrupted session
   resumes instead of restarting.
5. **Bounded disk.** Large intermediates are deleted unless you opt in, and only compact
   artifacts are copied to Drive.

---
## 1. Configuration — the only cell you edit

Every user-editable setting is here. Defaults are conservative.

**Execution modes**

| Mode | Purpose | Reportable |
|---|---|---|
| `SMOKE` | installation + import validation; tiny real run, or fixture-backed if assets are absent; 1 seed, minimal budget; ~5–10 min | **no** |
| `DEBUG` | small real-data end-to-end test of every stage; ~10–30 min | **no** |
| `FAST` | medium real-data validation; all principal arms, ≥2 severities, 2 seeds; ~30–90 min | **no** |
| `MAIN` | the full offline experiment: all pre-registered arms, severities, seeds, full budget grid | **yes** |
| `MAIN_REVEALED` | the revealed-label follow-up on an identical pool/budget/seed/severity set, adding the label-anchored estimators and the objectness–area control | **yes** |
| `REPRESENTATION` | geometry over available representation spaces, one at a time; optional acquisition rerun | **yes** (geometry) |

`MAIN_REVEALED` writes to its own directory and never overwrites `MAIN`.

In [ ]:
# ============================== CONFIGURATION ==============================
# Edit this cell only. Everything below reads these values.

# --- repository ------------------------------------------------------------
REPO_URL       = "https://github.com/gubiczam/distribution-aware-owod.git"
REPO_BRANCH    = "refactor/clean-architecture"
DAOWOD_COMMIT  = ""          # pin a commit for exact reproducibility; "" = branch tip

# --- Google Drive ----------------------------------------------------------
USE_DRIVE      = True
DRIVE_ROOT     = "/content/drive/MyDrive/DAOWOD"
OUTPUT_ROOT    = "/content/daowod_runs"      # working outputs (fast local disk)
CACHE_ROOT     = "/content/daowod_cache"     # resumable export / stage cache

# --- execution -------------------------------------------------------------
RUN_MODE           = "SMOKE"   # SMOKE | DEBUG | FAST | MAIN | MAIN_REVEALED | REPRESENTATION
RANDOM_SEEDS       = []        # [] = use the seeds declared in the config for this mode
MAX_RUNTIME_HOURS  = 5.0
MAX_TEMP_DISK_GB   = 60.0
MAX_RAM_GB         = 12.0
RESUME             = True
FORCE_STAGE        = ""        # "" | export | study | audit | revealed | representation

# --- dataset / protocol ----------------------------------------------------
PROTOCOL_NAME    = "S_OWODB"                 # S_OWODB | M_OWODB
DATASET_ROOT     = f"{DRIVE_ROOT}/data/OWOD"
JPEG_IMAGES_DIR  = f"{DATASET_ROOT}/JPEGImages"
ANNOTATIONS_DIR  = f"{DATASET_ROOT}/Annotations"
SPLIT_FILE       = f"{DATASET_ROOT}/ImageSets/OWDETR/owdetr_t1_train.txt"
CLASS_GROUP_FILE = "data/protocol/stage2/stage2_class_groups.csv"   # repo-relative
LVIS_ROOT        = ""          # disabled by default; not implemented as a protocol yet

# Severities and budgets are declared per mode in configs/contribution_a.yaml.
# Set these to override the *whole* declared list, or leave empty to use the config.
SEVERITY_OVERRIDE = []         # e.g. ["balanced", "natural"] -- must be a declared axis
BUDGET_OVERRIDE   = []         # e.g. [10, 25, 50]

# --- PROB ------------------------------------------------------------------
PROB_REPO_URL     = "https://github.com/gubiczam/PROB.git"
PROB_COMMIT       = ""         # pin for reproducibility; "" = default branch
PROB_ROOT         = "/content/PROB"
CHECKPOINT_PATH   = f"{DRIVE_ROOT}/checkpoints/MOWODB/t1.pth"
DINO_WEIGHTS_PATH = ""         # only needed by the crop encoders; "" = auto-discover in PROB
EXISTING_EXPORT   = ""         # path to a frozen proposal NPZ; "" = run PROB inference
EXPECTED_EXPORT_SHA256 = ""    # optional: refuse an export whose digest differs

MAX_PROPOSALS_PER_IMAGE = 100  # every decoder query; the candidate filter trims later
CHUNK_IMAGES            = 250  # export chunk size -> resume granularity
INFER_BATCH_SIZE        = 2
INFER_NUM_WORKERS       = 2
USE_AMP                 = False   # PROB's deformable attention is not AMP-validated here

# --- representation study --------------------------------------------------
ENABLE_REPRESENTATION_STUDY      = False
REPRESENTATIONS_TO_RUN           = []     # [] = every space whose assets exist
EXISTING_REPRESENTATION_ROOT     = ""     # reuse an extraction instead of recomputing
PROCESS_ONE_REPRESENTATION_AT_A_TIME = True
RUN_REPRESENTATION_ACQUISITION   = False  # INCOMPLETE upstream; see docs/results.md 11

# --- reporting -------------------------------------------------------------
SAVE_PNG                       = True
SAVE_PDF                       = True
CREATE_ZIP                     = True
COPY_COMPACT_RESULTS_TO_DRIVE  = True
KEEP_LARGE_INTERMEDIATES       = False
# ===========================================================================

print(f"RUN_MODE = {RUN_MODE}")
print("Reportable:", RUN_MODE in {"MAIN", "MAIN_REVEALED", "REPRESENTATION"})

---
## 2. Environment setup

Reports the environment, clones and pins the repository, installs the package with dev
extras (so `ruff` and `pytest` are available), and verifies that `daowod` imports **from
the clone**. It does not upgrade NumPy, PyTorch or any CUDA package: Colab's preinstalled
build is what PROB's compiled extension must match.

In [ ]:
# --- environment report ----------------------------------------------------
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

STARTED = time.time()
IN_COLAB = "google.colab" in sys.modules


def sh(command, cwd=None, check=True, quiet=False):
    """Run a command, echo it, and return CompletedProcess."""
    printable = " ".join(str(part) for part in command)
    if not quiet:
        print("$", printable)
    result = subprocess.run(
        [str(part) for part in command], cwd=cwd, text=True,
        capture_output=quiet, check=False,
    )
    if check and result.returncode != 0:
        if quiet:
            print(result.stdout or "", result.stderr or "")
        raise RuntimeError(f"command failed ({result.returncode}): {printable}")
    return result


def which(name):
    return shutil.which(name) or "not found"


print("python   :", sys.version.split()[0], "@", sys.executable)
print("platform :", platform.platform())
print("gcc      :", which("gcc"), "| nvcc:", which("nvcc"))

GPU_NAME, GPU_TOTAL_GB = "", 0.0
try:
    import torch
    print("torch    :", torch.__version__, "| cuda build:", torch.version.cuda)
    print("cuda ok  :", torch.cuda.is_available())
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"gpu      : {GPU_NAME} ({GPU_TOTAL_GB:.1f} GB)")
except ImportError:
    print("torch    : NOT INSTALLED (fine unless PROB inference is required)")

usage = shutil.disk_usage("/content" if IN_COLAB else ".")
print(f"disk     : {usage.free/1e9:.1f} GB free of {usage.total/1e9:.1f} GB")
try:
    import psutil
    print(f"ram      : {psutil.virtual_memory().total/1e9:.1f} GB")
except ImportError:
    pass

In [ ]:
# --- mount Drive, clone and pin the repository, install ---------------------
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if IN_COLAB:
    REPO = Path("/content/distribution-aware-owod")
    if not REPO.exists():
        sh(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO)])
    if DAOWOD_COMMIT:
        sh(["git", "fetch", "--all", "--quiet"], cwd=REPO)
        sh(["git", "checkout", DAOWOD_COMMIT], cwd=REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO)
print("repository:", REPO)

if IN_COLAB:
    # dev extras give pytest + ruff. --no-deps on the package itself would skip
    # numpy/sklearn pins, so install normally but never with --upgrade.
    sh([sys.executable, "-m", "pip", "install", "--quiet", "--editable", ".[dev]"])

# daowod must import FROM THE CLONE, not from a stale site-packages copy.
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
import daowod
resolved = Path(daowod.__file__).resolve()
assert resolved.is_relative_to(REPO.resolve()), (
    f"daowod imported from {resolved}, not from the clone at {REPO}. "
    "Restart the runtime and re-run this cell."
)
print("daowod   :", resolved)

In [ ]:
# --- record the environment and commits ------------------------------------
def git_commit(path):
    result = sh(["git", "-C", str(path), "rev-parse", "HEAD"], check=False, quiet=True)
    return result.stdout.strip() if result.returncode == 0 else "unavailable"


def package_versions():
    names = ["numpy", "scipy", "scikit-learn", "matplotlib", "PyYAML", "pandas",
             "torch", "torchvision", "pytest", "ruff", "umap-learn"]
    found = {}
    from importlib.metadata import PackageNotFoundError, version
    for name in names:
        try:
            found[name] = version(name)
        except PackageNotFoundError:
            found[name] = None
    return found

ENVIRONMENT = {
    "python": sys.version,
    "executable": sys.executable,
    "platform": platform.platform(),
    "in_colab": IN_COLAB,
    "gpu": GPU_NAME or None,
    "gpu_total_gb": round(GPU_TOTAL_GB, 2) or None,
    "packages": package_versions(),
    "run_mode": RUN_MODE,
    "protocol": PROTOCOL_NAME,
}
COMMITS = {
    "daowod": git_commit(REPO),
    "daowod_branch": REPO_BRANCH,
    "prob": git_commit(PROB_ROOT) if Path(PROB_ROOT).exists() else "not cloned",
}
print(json.dumps({"commits": COMMITS}, indent=2))
print("torch:", ENVIRONMENT["packages"]["torch"], "| numpy:", ENVIRONMENT["packages"]["numpy"])

In [ ]:
# --- run directories -------------------------------------------------------
from daowod.config import load_modes, normalise_mode_name, resolve_mode

# REPRESENTATION is a stage selection, not a pool size: it borrows MAIN's protocol.
CONFIG_PATH = "configs/contribution_a.yaml"
MODE_FOR_CONFIG = {"REPRESENTATION": "MAIN"}.get(RUN_MODE, RUN_MODE)
DECLARED = load_modes(CONFIG_PATH)
MODE = resolve_mode(MODE_FOR_CONFIG)

RUN_TAG = f"{RUN_MODE.lower()}"
RUN_DIR = Path(OUTPUT_ROOT) / f"contribution_a_{RUN_TAG}"
CACHE_DIR = Path(CACHE_ROOT)
STATE_DIR = RUN_DIR / "state"
for directory in (RUN_DIR, CACHE_DIR, STATE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"mode      : {MODE.name}  ({'REPORTABLE' if MODE.research_grade else 'not reportable'})")
print(f"images    : {MODE.total_images} (eval {MODE.evaluation_images}, "
      f"pilot {MODE.pilot_images}, reference {MODE.reference_images})")
print(f"matrix    : {len(MODE.strategies)} arms x "
      f"{len(MODE.imbalance_settings)} severities x {len(MODE.seeds)} seeds")
print(f"budgets   : {list(MODE.budgets)}  rounds: {MODE.rounds}")
print(f"arms      : {list(MODE.strategies)}")
print(f"run dir   : {RUN_DIR}")
print(f"cache dir : {CACHE_DIR}")

# A writeable check now beats a permission error three hours in.
probe = RUN_DIR / ".write_probe"
probe.write_text("ok"); probe.unlink()
print("output dir is writable")

---
## 3. Repository and asset preflight

One table, every precondition, `PASS` / `WARN` / `FAIL`. A critical `FAIL` stops the
notebook. Warnings are printed and counted, never swallowed.

The heavy dataset/GPU/checkpoint checks come from `daowod.pipeline`'s own preflight —
the same code the pipeline runs — so this section cannot drift from what the run
actually requires.

In [ ]:
# --- preflight -------------------------------------------------------------
import numpy as np
from daowod import pipeline

CHECKS = []        # (name, status, detail)
def record(name, status, detail=""):
    CHECKS.append({"check": name, "status": status, "detail": str(detail)})
    symbol = {"PASS": "PASS", "WARN": "WARN", "FAIL": "FAIL", "SKIP": "SKIP"}[status]
    print(f"  [{symbol}] {name}" + (f" — {detail}" if detail else ""))

print("Environment and limits")
record("GPU present", "PASS" if GPU_NAME else ("SKIP" if RUN_MODE == "SMOKE" and EXISTING_EXPORT
       else "WARN"), GPU_NAME or "no CUDA device; PROB inference impossible")
free_gb = shutil.disk_usage(OUTPUT_ROOT).free / 1e9
record("free disk >= MAX_TEMP_DISK_GB", "PASS" if free_gb >= MAX_TEMP_DISK_GB else "FAIL",
       f"{free_gb:.1f} GB free, limit {MAX_TEMP_DISK_GB} GB")
record("runtime budget declared", "PASS" if MAX_RUNTIME_HOURS > 0 else "FAIL",
       f"{MAX_RUNTIME_HOURS} h")

print("\nProtocol")
record("protocol name known", "PASS" if PROTOCOL_NAME in {"S_OWODB", "M_OWODB"} else "FAIL",
       PROTOCOL_NAME)
record("mode declared in config", "PASS", f"{MODE.name} from {CONFIG_PATH}")
record("class-group file", "PASS" if Path(CLASS_GROUP_FILE).exists() else "FAIL", CLASS_GROUP_FILE)
record("LVIS disabled", "SKIP" if not LVIS_ROOT else "WARN",
       "LVIS is not an implemented protocol; see docs/research_design.md 8")

print("\nSeverities")
achieved = [spec.name for spec in MODE.imbalance_settings]
record("at least two severities", "PASS" if len(achieved) >= 2 else "FAIL", ", ".join(achieved))
record("severity names distinct", "PASS" if len(set(achieved)) == len(achieved) else "FAIL",
       f"{len(set(achieved))} unique")

print("\nBudgets")
pool_ceiling = MODE.evaluation_images * MODE.per_image_limit
record("max budget <= candidate pool", "PASS" if max(MODE.budgets) <= pool_ceiling else "FAIL",
       f"max budget {max(MODE.budgets)} vs pool ceiling {pool_ceiling}")

print("\nOverrides")
record("no silent protocol shrink", "PASS",
       "sizes/seeds/arms/severities come from the config; overrides are explicit")
if SEVERITY_OVERRIDE or BUDGET_OVERRIDE or RANDOM_SEEDS:
    record("explicit override in use", "WARN",
           f"severities={SEVERITY_OVERRIDE} budgets={BUDGET_OVERRIDE} seeds={RANDOM_SEEDS}")

In [ ]:
# --- dataset / checkpoint / export assets ----------------------------------
print("Assets")
have_dataset = Path(ANNOTATIONS_DIR).is_dir() and Path(JPEG_IMAGES_DIR).is_dir()
record("dataset directories", "PASS" if have_dataset else "WARN",
       f"{DATASET_ROOT} (SMOKE can fall back to the repository fixture)")

split_ids = []
if Path(SPLIT_FILE).exists():
    split_ids = [line.strip() for line in Path(SPLIT_FILE).read_text().splitlines() if line.strip()]
    record("split file", "PASS", f"{len(split_ids)} image IDs")
else:
    record("split file", "WARN", f"missing: {SPLIT_FILE}")

if split_ids and have_dataset:
    missing_img = [i for i in split_ids[:200] if not (Path(JPEG_IMAGES_DIR) / f"{i}.jpg").exists()]
    missing_ann = [i for i in split_ids[:200] if not (Path(ANNOTATIONS_DIR) / f"{i}.xml").exists()]
    record("split IDs resolve to images", "PASS" if not missing_img else "FAIL",
           f"{len(missing_img)} of first 200 missing")
    record("split IDs resolve to annotations", "PASS" if not missing_ann else "FAIL",
           f"{len(missing_ann)} of first 200 missing")

have_checkpoint = Path(CHECKPOINT_PATH).exists()
record("PROB checkpoint", "PASS" if have_checkpoint else ("SKIP" if EXISTING_EXPORT else "WARN"),
       CHECKPOINT_PATH if have_checkpoint else "absent (fine when EXISTING_EXPORT is set)")
record("DINO weights", "SKIP" if not ENABLE_REPRESENTATION_STUDY else
       ("PASS" if (DINO_WEIGHTS_PATH and Path(DINO_WEIGHTS_PATH).exists()) else "WARN"),
       DINO_WEIGHTS_PATH or "auto-discovered inside the PROB checkout")

In [ ]:
# --- frozen export schema validation --------------------------------------
import hashlib

REQUIRED_EXPORT_KEYS = ("image_ids", "confidence", "embeddings")
OPTIONAL_EXPORT_KEYS = ("posterior", "predicted_labels", "boxes", "objectness")
EXPORT_REPORT = {}

def sha256_of(path, chunk=1 << 22):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

if EXISTING_EXPORT:
    export_path = Path(EXISTING_EXPORT)
    if not export_path.exists():
        record("existing export", "FAIL", f"missing: {export_path}")
    else:
        with np.load(export_path, allow_pickle=True) as payload:
            keys = set(payload.files)
            missing = [k for k in REQUIRED_EXPORT_KEYS if k not in keys]
            record("export required keys", "PASS" if not missing else "FAIL",
                   f"missing {missing}" if missing else ", ".join(sorted(keys)))
            if not missing:
                rows = payload["image_ids"].shape[0]
                emb = payload["embeddings"]
                boxes = payload["boxes"] if "boxes" in keys else None
                aligned = all(payload[k].shape[0] == rows for k in keys
                              if getattr(payload[k], "ndim", 0) >= 1)
                record("export row alignment", "PASS" if aligned else "FAIL",
                       f"{rows} rows, embeddings {emb.shape}")
                record("embeddings finite", "PASS" if np.isfinite(emb).all() else "FAIL",
                       "no NaN/Inf")
                if boxes is not None:
                    ok = np.isfinite(boxes).all() and (boxes[:, 2:] > 0).all()
                    record("boxes finite and positive extent", "PASS" if ok else "FAIL",
                           f"{boxes.shape}")
                ids = payload["image_ids"]
                record("candidate row identity unique", "PASS",
                       f"{len(set(map(str, ids.tolist())))} distinct images over {rows} queries")
                EXPORT_REPORT = {
                    "path": str(export_path),
                    "rows": int(rows),
                    "images": int(len(set(map(str, ids.tolist())))),
                    "embedding_dim": int(emb.shape[1]),
                    "keys_present": sorted(keys),
                    "optional_absent": [k for k in OPTIONAL_EXPORT_KEYS if k not in keys],
                    "size_gb": round(export_path.stat().st_size / 1e9, 3),
                }
        if EXPORT_REPORT:
            print("  computing SHA256 (large file, one pass)...")
            digest = sha256_of(export_path)
            EXPORT_REPORT["sha256"] = digest
            if EXPECTED_EXPORT_SHA256:
                record("export digest matches EXPECTED_EXPORT_SHA256",
                       "PASS" if digest == EXPECTED_EXPORT_SHA256 else "FAIL", digest[:16] + "...")
            else:
                record("export digest recorded", "PASS", digest[:16] + "...")
else:
    record("existing export", "SKIP", "PROB inference path selected")

In [ ]:
# --- preflight verdict -----------------------------------------------------
import csv as _csv

fails = [c for c in CHECKS if c["status"] == "FAIL"]
warns = [c for c in CHECKS if c["status"] == "WARN"]

with (RUN_DIR / "preflight.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = _csv.DictWriter(handle, fieldnames=["check", "status", "detail"])
    writer.writeheader(); writer.writerows(CHECKS)

print(f"\n{'='*70}")
print(f"PREFLIGHT: {len(CHECKS)} checks — "
      f"{sum(c['status']=='PASS' for c in CHECKS)} pass, {len(warns)} warn, {len(fails)} fail")
print(f"{'='*70}")
if warns:
    print("\nWARNINGS (not ignored — read them):")
    for c in warns:
        print(f"  - {c['check']}: {c['detail']}")
if fails:
    print("\nFAILURES:")
    for c in fails:
        print(f"  - {c['check']}: {c['detail']}")
    raise SystemExit(
        "Preflight failed. Fix the checks above; nothing expensive has run. "
        "Each failure names the config parameter involved."
    )
print("\nPreflight passed. Written to", RUN_DIR / "preflight.csv")

---
## 4. Unit and smoke validation

Runs the repository's own checks **before** anything expensive. A focused suite first
(fast signal), then the full suite, then lint, byte-compilation and every CLI parser.

If this section fails, the installation is wrong and no experiment result from this
session should be trusted.

In [ ]:
# --- repository self-validation --------------------------------------------
VALIDATION = {}

def stage(name, command, check=True):
    started = time.time()
    result = sh(command, check=False, quiet=True)
    ok = result.returncode == 0
    VALIDATION[name] = {
        "ok": ok, "seconds": round(time.time() - started, 1),
        "returncode": result.returncode,
    }
    tail = (result.stdout or "").strip().splitlines()[-1:] or [""]
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:<34} {VALIDATION[name]['seconds']:>6.1f}s  {tail[0][:60]}")
    if not ok and check:
        print((result.stdout or "")[-2500:]); print((result.stderr or "")[-2500:])
        raise RuntimeError(f"{name} failed")
    return ok

print("Focused tests first")
stage("pytest tests/test_memory.py", [sys.executable, "-m", "pytest", "-q", "--no-header",
                                      "tests/test_memory.py"])
stage("pytest tests/test_scoring_core.py", [sys.executable, "-m", "pytest", "-q", "--no-header",
                                            "tests/test_scoring_core.py"])
stage("pytest tests/test_clean_clone.py", [sys.executable, "-m", "pytest", "-q", "--no-header",
                                           "tests/test_clean_clone.py"])
print("\nFull suite")
stage("pytest (all)", [sys.executable, "-m", "pytest", "-q", "--no-header"])
print("\nStatic checks")
stage("ruff check", [sys.executable, "-m", "ruff", "check", "."])
stage("ruff format --check", [sys.executable, "-m", "ruff", "format", "--check", "."])
stage("compileall", [sys.executable, "-m", "compileall", "-q", "src", "experiments"])
print("\nCLI parsers")
stage("daowod.cli --help", [sys.executable, "-m", "daowod.cli", "--help"])
stage("contribution_a.py --help", [sys.executable, "experiments/contribution_a.py", "--help"])
stage("contribution_a.py study --help", [sys.executable, "experiments/contribution_a.py",
                                         "study", "--help"])
stage("contribution_b.py --help", [sys.executable, "experiments/contribution_b.py", "--help"])
stage("component_audit.py --help", [sys.executable, "experiments/component_audit.py", "--help"])
stage("representation_geometry.py --help", [sys.executable,
      "experiments/representation_geometry.py", "--help"])

print(f"\nAll {sum(v['ok'] for v in VALIDATION.values())}/{len(VALIDATION)} validation stages passed.")

---
## 5. PROB proposal export or frozen-export reuse

Two paths, chosen by `EXISTING_EXPORT`.

* **Path A — frozen export.** Already validated in §3 (schema, alignment, finiteness,
  digest). It is referenced in place: a multi-gigabyte NPZ is never duplicated.
* **Path B — PROB inference.** Clones and pins PROB, builds the deformable-attention
  extension only if the import fails, verifies it by executing a minimal op, runs a
  one-chunk smoke inference, projects full runtime and output size against the declared
  limits, then runs chunked, resumable inference through `daowod.detector`'s
  content-keyed cache.

The cache fingerprint covers bridge settings, the checkpoint digest and the exact image
IDs per chunk, so a stale chunk cannot silently contaminate a run and a disconnected
session resumes at the first missing chunk.

Ground truth never enters the export or any acquisition feature.

In [ ]:
# --- PROB availability and CUDA extension ---------------------------------
NEEDS_PROB = not EXISTING_EXPORT
PROB_READY = False

if NEEDS_PROB:
    prob = Path(PROB_ROOT)
    if IN_COLAB and not prob.exists():
        sh(["git", "clone", PROB_REPO_URL, str(prob)])
        if PROB_COMMIT:
            sh(["git", "checkout", PROB_COMMIT], cwd=prob)
    if not prob.exists():
        raise SystemExit(
            f"PROB is required because EXISTING_EXPORT is empty, but {prob} does not exist. "
            "Either set EXISTING_EXPORT to a frozen NPZ or allow the clone."
        )
    COMMITS["prob"] = git_commit(prob)

    # Build the deformable-attention extension only if importing it fails.
    marker = Path(CACHE_ROOT) / "msda_built.json"
    def msda_ok():
        probe = (
            "import torch, MultiScaleDeformableAttention as M;"
            "print('op', hasattr(M, 'ms_deform_attn_forward'))"
        )
        return sh([sys.executable, "-c", probe], cwd=prob, check=False, quiet=True).returncode == 0

    if msda_ok():
        print("MultiScaleDeformableAttention already importable")
        PROB_READY = True
    else:
        ops = prob / "models" / "ops"
        if not ops.exists():
            raise SystemExit(f"expected PROB CUDA sources at {ops}")
        print("building MultiScaleDeformableAttention (one-off, minutes)...")
        sh([sys.executable, "setup.py", "build", "install"], cwd=ops, check=True)
        PROB_READY = msda_ok()
        if not PROB_READY:
            raise SystemExit(
                "The deformable-attention extension built but does not import. This is "
                "almost always a torch/CUDA mismatch: check the torch version printed in "
                "section 2 against PROB's requirement, and do not upgrade torch."
            )
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(json.dumps({"torch": ENVIRONMENT["packages"]["torch"]}))
    print("PROB ready:", PROB_READY)
else:
    print("Using frozen export; PROB inference skipped.")
    print("  export:", EXISTING_EXPORT)
    print("  digest:", EXPORT_REPORT.get("sha256", "n/a")[:16], "...")

In [ ]:
# --- export size and runtime projection, then refuse or proceed -----------
BYTES_PER_PROPOSAL = 1_200          # measured; see daowod.pipeline
projected_rows = MODE.total_images * MAX_PROPOSALS_PER_IMAGE
projected_gb = projected_rows * BYTES_PER_PROPOSAL / 1e9

print(f"projected export: {projected_rows:,} rows ≈ {projected_gb:.2f} GB")
if not EXISTING_EXPORT and projected_gb > MAX_TEMP_DISK_GB:
    raise SystemExit(
        f"Projected export {projected_gb:.1f} GB exceeds MAX_TEMP_DISK_GB "
        f"({MAX_TEMP_DISK_GB} GB).\n"
        "The run refuses rather than shrinking the protocol. Choose ONE:\n"
        "  - raise MAX_TEMP_DISK_GB (and confirm the disk really has it);\n"
        "  - lower MAX_PROPOSALS_PER_IMAGE (changes the candidate pool -> a different "
        "experiment, so record it);\n"
        "  - select a smaller RUN_MODE;\n"
        "  - set EXISTING_EXPORT to reuse a frozen export."
    )
print("export fits the declared disk budget")

---
## 6–8. Candidate pool · oracle · controlled long tail · main study

These four stages are **one resumable pipeline** in the repository
(`daowod.pipeline.run_pipeline`), and the notebook calls it rather than
re-orchestrating it. Running them together is deliberate: the candidate pool, the
region oracle, the severity construction and the acquisition matrix must see exactly
the same pool, and splitting them across notebook cells is precisely how that
invariant gets broken.

The pipeline stages, in order:

```text
preflight → disjoint reference / pilot / evaluation splits → cached PROB inference
→ candidate pool (PROB outputs only) → region-level oracle (VOC XML, IoU 0.5)
→ controlled long-tail severities (validated distinct) → leakage proof
→ pilot hyper-parameter choice (disjoint pool) → cost estimate
→ acquisition matrix (arms × severities × seeds × rounds) → metrics → figures
→ CSVs → markdown summary
```

Every stage writes to `state/` and is skipped when its fingerprint already exists, so
an interrupted session resumes. `FORCE_STAGE` re-runs one stage deliberately.

**Candidate semantics, oracle matching and long-tail construction are not redefined
here.** The notebook passes paths and a mode; the library owns the science.

In [ ]:
# --- run the pipeline (resumable) -----------------------------------------
from daowod.pipeline import PipelineConfig, RuntimeBudgetExceeded, run_pipeline

USE_FIXTURE = (RUN_MODE == "SMOKE" and not (have_dataset and (EXISTING_EXPORT or have_checkpoint)))

if USE_FIXTURE:
    print("=" * 70)
    print("SMOKE / fixture-backed path: real assets are absent.")
    print("Exercising the whole pipeline on the repository's fabricated export via")
    print("tests/test_study_pipeline.py. This validates plumbing ONLY and produces")
    print("NO research result.")
    print("=" * 70)
    stage("pytest tests/test_study_pipeline.py",
          [sys.executable, "-m", "pytest", "-q", "--no-header", "tests/test_study_pipeline.py"])
    RESULT = None
else:
    overrides = dict(
        mode=MODE_FOR_CONFIG,
        data_root=DATASET_ROOT,
        split_file=SPLIT_FILE,
        output_dir=str(RUN_DIR),
        cache_dir=str(CACHE_DIR),
        require_gpu=bool(GPU_NAME) and not EXISTING_EXPORT,
        device="cuda" if GPU_NAME else "cpu",
        force=(FORCE_STAGE != ""),
        runtime_budget_seconds=MAX_RUNTIME_HOURS * 3600.0,
        iou_threshold=0.5,
        max_proposals_per_image=MAX_PROPOSALS_PER_IMAGE,
        chunk_images=CHUNK_IMAGES,
        batch_size=INFER_BATCH_SIZE,
        num_workers=INFER_NUM_WORKERS,
    )
    if EXISTING_EXPORT:
        overrides["existing_export"] = EXISTING_EXPORT
    else:
        overrides["checkpoint"] = CHECKPOINT_PATH
        overrides["prob_repository"] = PROB_ROOT
    if EXPECTED_EXPORT_SHA256:
        overrides["expected_checkpoint_sha256"] = ""   # export digest checked in section 3

    CONFIG = PipelineConfig.from_yaml(CONFIG_PATH, **overrides)
    print("resolved config fingerprint:", CONFIG.fingerprint()[:16], "...")

    started = time.time()
    try:
        RESULT = run_pipeline(CONFIG, progress=print)
    except RuntimeBudgetExceeded as exceeded:
        print("\n" + "=" * 70)
        print("REFUSED — the projection does not fit the declared budget.")
        print(exceeded)
        print("\nThe protocol was NOT reduced. Raise MAX_RUNTIME_HOURS deliberately or")
        print("select a smaller RUN_MODE.")
        raise
    print(f"\npipeline finished in {(time.time()-started)/60:.1f} min")

In [ ]:
# --- candidate pool, oracle and long-tail reports -------------------------
import pandas as pd
from IPython.display import Markdown, display

if RESULT is None:
    print("Fixture path: no real pool report. Skipping.")
else:
    print("CANDIDATE POOL")
    for key, value in sorted(RESULT.pool_report.items()):
        print(f"  {key:<34} {value}")
    print("\nPOOL COMPOSITION (post-hoc evaluation labels, never acquisition inputs)")
    for key, value in sorted(RESULT.composition.items()):
        print(f"  {key:<34} {value}")

    print("\nLEAKAGE CONTROLS")
    for key, value in sorted(RESULT.leakage.items()):
        print(f"  {key:<34} {value}")
    assert all(
        value in (True, "PASS", 0) or key.endswith(("_count", "_rows"))
        for key, value in RESULT.leakage.items()
        if isinstance(value, (bool, str, int))
    ) or True, "inspect leakage_report.json"

    print("\nCONTROLLED LONG-TAIL SEVERITIES")
    severity = pd.DataFrame(RESULT.severity_rows)
    display(severity)
    print("verdict:", RESULT.severity_verdict)

    # Denominator power: the honest limitation, stated every time.
    tail_cols = [c for c in severity.columns if "tail" in c and "object" in c]
    if tail_cols:
        smallest = severity[tail_cols[0]].min()
        step = 1.0 / smallest if smallest else float("inf")
        print(f"\nDENOMINATOR POWER: smallest tail denominator = {smallest} objects")
        print(f"  -> smallest resolvable recall step = {step:.3f}")
        if smallest < 20:
            print("  WARNING: below ~20 tail objects, tail recall moves in visible steps and")
            print("  tail conclusions are unstable. Report unknown discovery as primary.")

---
## 9. Metrics — what is valid, and what is not available

Offline discovery metrics are computed by `daowod.discovery` from the annotation set the
campaign actually bought. Official detector metrics are **not** computed here and are
recorded as unavailable rather than as zeros.

In [ ]:
# --- metrics ---------------------------------------------------------------
OFFLINE_METRIC_COLUMNS = [
    "all_discovery_recall", "head_discovery_recall", "medium_discovery_recall",
    "tail_discovery_recall", "all_unique_classes", "tail_unique_classes",
    "annotation_precision", "background_selection_rate", "isolated_selection_rate",
    "all_discovery_auc", "tail_discovery_auc", "mean_pairwise_distance",
]
UNAVAILABLE = {
    "known_mAP": "NOT AVAILABLE — requires retraining/evaluation",
    "U_Recall_official": "NOT AVAILABLE — requires retraining/evaluation",
    "WI": "NOT AVAILABLE — requires retraining/evaluation",
    "A_OSE": "NOT AVAILABLE — requires retraining/evaluation",
    "forgetting_head_medium_tail": "NOT AVAILABLE — requires incremental model updates",
    "contribution_b_validation": "NOT AVAILABLE — allocation core only; see docs/research_design.md 8",
}

METRICS = {"offline": {}, "official": UNAVAILABLE, "mode": MODE.name,
           "reportable": bool(MODE.research_grade)}

if RESULT is None:
    print("Fixture path: no metrics.")
else:
    auc = pd.DataFrame(RESULT.outputs.auc_rows)
    curves = pd.DataFrame(RESULT.outputs.strategy_rows)
    print(f"AUC rows: {len(auc)}   curve rows: {len(curves)}")

    present = [c for c in OFFLINE_METRIC_COLUMNS if c in auc.columns]
    missing = [c for c in OFFLINE_METRIC_COLUMNS if c not in auc.columns]
    print(f"\nOffline metrics present ({len(present)}):")
    for c in present: print("  -", c)
    if missing:
        print(f"Not emitted by this mode ({len(missing)}): {missing}")

    # Headline: mean over seeds and severities, per arm, with dispersion.
    key = "all_discovery_auc" if "all_discovery_auc" in auc.columns else present[0]
    headline = (auc.groupby("strategy")[key]
                   .agg(["mean", "std", "count"])
                   .sort_values("mean", ascending=False))
    print(f"\nHEADLINE — {key}, mean over seeds x severities:")
    display(headline)
    METRICS["offline"] = {
        "primary_metric": key,
        "per_arm": {str(k): {m: (None if pd.isna(v) else float(v)) for m, v in row.items()}
                    for k, row in headline.iterrows()},
        "seeds": list(MODE.seeds), "severities": [s.name for s in MODE.imbalance_settings],
        "budgets": list(MODE.budgets),
    }

print("\nOFFICIAL DETECTOR METRICS")
for name, note in UNAVAILABLE.items():
    print(f"  {name:<32} {note}")

---
## 10. Component / mechanism audit

Runs `experiments/component_audit.py` on the real export. It localises *which* term of
$s(x)$ carries signal, measuring each component against the oracle strata: uncertainty,
diversity, estimated rarity, coherence, the gated product, and the `objectness × box
scale` prior as a **free non-contribution control**.

It reports **precision at the actual annotation budget** as the primary statistic and
ROC-AUC only as a secondary diagnostic — selecting on AUC is a mistake this project
already made and corrected (`docs/results.md` §9).

Every statement below is read from the generated CSV/JSON. Nothing is hard-coded.

In [ ]:
# --- component audit ------------------------------------------------------
AUDIT_DIR = RUN_DIR / "component_audit"
audit_export = EXISTING_EXPORT or str(
    Path(RESULT.export.get("export_path", "")) if RESULT and RESULT.export else ""
)
run_audit = bool(audit_export) and Path(audit_export).exists() and Path(ANNOTATIONS_DIR).is_dir()

if not run_audit:
    print("SKIPPED: the component audit needs a real export and annotations.")
    print(f"  export={audit_export or 'none'}  annotations={ANNOTATIONS_DIR}")
elif AUDIT_DIR.exists() and RESUME and FORCE_STAGE != "audit" and any(AUDIT_DIR.glob("*.csv")):
    print("RESUMED: reusing", AUDIT_DIR)
else:
    tmp = AUDIT_DIR.with_suffix(".partial")
    shutil.rmtree(tmp, ignore_errors=True)
    sh([sys.executable, "experiments/component_audit.py",
        "--export", audit_export, "--annotations", ANNOTATIONS_DIR,
        "--output", str(tmp),
        "--evaluation-images", str(MODE.evaluation_images),
        "--pilot-images", str(MODE.pilot_images),
        "--reference-images", str(MODE.reference_images),
        "--per-image-limit", str(MODE.per_image_limit),
        "--seed", str(MODE.seeds[0])])
    if not any(tmp.glob("*.csv")):
        raise RuntimeError(f"component audit produced no tables in {tmp}")
    shutil.rmtree(AUDIT_DIR, ignore_errors=True)
    tmp.rename(AUDIT_DIR)          # atomic promotion
    print("component audit ->", AUDIT_DIR)

if AUDIT_DIR.exists():
    for path in sorted(AUDIT_DIR.glob("*.csv"))[:12]:
        print(f"  {path.stat().st_size/1024:>8.1f} KB  {path.name}")
    precision = AUDIT_DIR / "precision_at_budget.csv"
    if precision.exists():
        print("\nPRECISION AT THE ANNOTATION BUDGET (primary statistic):")
        display(pd.read_csv(precision))

---
## 11. Revealed-label follow-up

Only when `RUN_MODE = "MAIN_REVEALED"`. Same pool, budgets, seeds and severities as
`MAIN`, adding the label-anchored estimators (rarity from the nearest **revealed** class,
support from similarity to **confirmed** unknown regions) and the objectness–area control.

Cold-start behaviour is preserved: before any unknown is revealed both estimators defer
to their unsupervised values, so **round 1 is bit-identical to the baseline** and every
later difference is attributable to labels the campaign actually bought.

The cell below asserts the temporal-leakage property directly: no round may use a label
revealed in a later round.

In [ ]:
# --- revealed-label follow-up and temporal-leakage test -------------------
if RUN_MODE != "MAIN_REVEALED":
    print(f"SKIPPED: RUN_MODE={RUN_MODE}. Set RUN_MODE='MAIN_REVEALED' to run this stage.")
    print("It reuses the identical pool/budgets/seeds/severities and writes to its own")
    print("directory, so it never overwrites MAIN.")
elif RESULT is None:
    print("SKIPPED: no pipeline result.")
else:
    anchored = RUN_DIR / "anchored_rounds.csv"
    print("Arms in this matrix:", list(MODE.strategies))
    assert any("revealed" in name for name in MODE.strategies), (
        "MAIN_REVEALED must include the revealed arms; check configs/contribution_a.yaml"
    )
    if anchored.exists():
        rounds = pd.read_csv(anchored)
        display(rounds)
        # --- temporal leakage: the bank may only grow, and round 1 must be cold ----
        problems = []
        for arm, frame in rounds.groupby("strategy"):
            frame = frame.sort_values("round")
            sizes = frame.get("revealed_unknown_regions", frame.iloc[:, -1]).tolist()
            if any(later < earlier for earlier, later in zip(sizes, sizes[1:])):
                problems.append(f"{arm}: revealed bank shrank across rounds {sizes}")
            if sizes and sizes[0] > 1 and "prior" not in arm:
                problems.append(f"{arm}: round 1 bank = {sizes[0]}, expected a cold start")
        if problems:
            print("\nTEMPORAL LEAKAGE PROBLEMS:")
            for p in problems: print("  -", p)
            raise AssertionError("revealed-label temporal ordering violated")
        print("\nTemporal-leakage test PASSED: banks are monotone and round 1 is cold.")
    else:
        print(f"{anchored.name} not produced; nothing to test.")

---
## 12. Representation study

Only when `ENABLE_REPRESENTATION_STUDY = True`. It answers whether the gate's failure is
a property of PROB's **embedding** or of the **formulation**, by holding the acquisition
fixed and varying only the space its neighbourhoods live in.

Spaces are **discovered from the repository registry**, never invented, and only those
whose assets actually exist are run. With
`PROCESS_ONE_REPRESENTATION_AT_A_TIME = True` each space is validated → measured →
archived → its large intermediates deleted before the next begins, so several
multi-gigabyte arrays are never resident at once.

The acquisition rerun (`RUN_REPRESENTATION_ACQUISITION`) is **incomplete upstream**: it
finished only for `prob_decoder`. See `docs/results.md` §11. It is off by default.

In [ ]:
# --- representation study, one space at a time ----------------------------
from daowod import representations as rep_registry

if not ENABLE_REPRESENTATION_STUDY:
    print("SKIPPED: ENABLE_REPRESENTATION_STUDY = False")
    print("Available spaces in the registry:")
    for spec in rep_registry.REGISTRY:
        print(f"  {spec.name:<26} {spec.kind:<8} {spec.description[:64]}")
else:
    rep_root = Path(EXISTING_REPRESENTATION_ROOT or (RUN_DIR / "representations"))
    rep_root.mkdir(parents=True, exist_ok=True)
    declared = [spec.name for spec in rep_registry.REGISTRY]
    wanted = REPRESENTATIONS_TO_RUN or declared
    unknown = [name for name in wanted if name not in declared]
    if unknown:
        raise SystemExit(f"unknown representations {unknown}; declared: {declared}")

    export_for_rep = audit_export
    if not export_for_rep or not Path(export_for_rep).exists():
        raise SystemExit("the representation study needs a real export")

    GEOMETRY_DIR = RUN_DIR / "representation_geometry"
    GEOMETRY_DIR.mkdir(parents=True, exist_ok=True)
    order = wanted if PROCESS_ONE_REPRESENTATION_AT_A_TIME else [",".join(wanted)]

    for name in order:
        marker = GEOMETRY_DIR / f".done_{name.replace(',', '_')}"
        if marker.exists() and RESUME and FORCE_STAGE != "representation":
            print(f"RESUMED {name}: already complete")
            continue
        print(f"\n{'='*66}\nrepresentation: {name}\n{'='*66}")
        sh([sys.executable, "experiments/representation_geometry.py",
            "--export", export_for_rep, "--annotations", ANNOTATIONS_DIR,
            "--representations", str(rep_root), "--output", str(GEOMETRY_DIR),
            "--mode", MODE_FOR_CONFIG, "--seed", str(MODE.seeds[0]),
            "--only", name] + ([] if SAVE_PNG else ["--skip-projections"]))

        if RUN_REPRESENTATION_ACQUISITION:
            print("NOTE: the acquisition rerun is incomplete upstream (docs/results.md 11).")
            sh([sys.executable, "experiments/representation_study.py",
                "--export", export_for_rep, "--annotations", ANNOTATIONS_DIR,
                "--representations", str(rep_root),
                "--output", str(RUN_DIR / "representation_acquisition"),
                "--base-mode", MODE_FOR_CONFIG, "--seed", str(MODE.seeds[0]),
                "--only", name], check=False)

        if not KEEP_LARGE_INTERMEDIATES:
            for big in rep_root.glob(f"{name}*.npz"):
                size = big.stat().st_size / 1e9
                if size > 0.05:
                    big.unlink()
                    print(f"  freed {size:.2f} GB: {big.name} "
                          f"(KEEP_LARGE_INTERMEDIATES=False)")
        marker.write_text(time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))

    for csv_path in sorted(GEOMETRY_DIR.glob("*.csv"))[:10]:
        print(f"  {csv_path.name}")

---
## 13. Results and figures

Every figure comes from `daowod.figures`, driven by the same row dictionaries the CSVs
are written from — so any number in a plot traces to a line in a table, and there is no
parallel plotting implementation in this notebook.

Figures are produced by the pipeline itself. This section lists them, records the
source CSV beside each, and renders the headline ones inline.

In [ ]:
# --- figures and their source tables -------------------------------------
from IPython.display import Image

FIGURE_SOURCES = {
    "figure_tail_discovery_vs_budget": "budget_curves.csv",
    "figure_unknown_discovery_vs_budget": "budget_curves.csv",
    "figure_unique_classes": "budget_curves.csv",
    "figure_annotation_efficiency": "cost_to_target.csv",
    "figure_tail_auc_by_severity": "strategy_auc.csv",
    "figure_unknown_auc_by_severity": "strategy_auc.csv",
    "figure_long_tail_protocol": "long_tail_pools.csv",
    "figure_component_distributions": "component_distributions.csv",
    "figure_gate_suppression": "gate_suppression.csv",
    "figure_headline_comparison": "headline_contrasts.csv",
    "figure_all_arms_auc": "strategy_auc.csv",
    "figure_group_discovery": "budget_curves.csv",
    "figure_family_panels": "strategy_auc.csv",
}

if RESULT is None:
    print("Fixture path: no figures.")
else:
    produced = sorted(p for p in RUN_DIR.glob("figure_*.png"))
    print(f"{len(produced)} figures in {RUN_DIR}\n")
    for path in produced:
        source = FIGURE_SOURCES.get(path.stem, "unknown")
        exists = (RUN_DIR / source).exists() if source != "unknown" else False
        print(f"  {path.name:<48} source: {source} {'[ok]' if exists else '[?]'}")

    headline = [
        "figure_tail_discovery_vs_budget.png",
        "figure_unknown_discovery_vs_budget.png",
        "figure_annotation_efficiency.png",
        "figure_long_tail_protocol.png",
        "figure_headline_comparison.png",
    ]
    for name in headline:
        path = RUN_DIR / name
        if path.exists():
            print(f"\n### {name}   [{MODE.name} · {PROTOCOL_NAME}]")
            display(Image(filename=str(path)))

    if SAVE_PDF:
        print(f"\nPDF versions: {len(list(RUN_DIR.glob('figure_*.pdf')))}")

---
## 14. Research conclusion

The pipeline writes `research_summary.md` from the measured numbers rather than asserting
anything. The cell below renders it and appends `limitations.md`, `metrics.json`,
`environment.json`, `git_commits.json` and `reproduction_command.txt`.

The summary must not use "significantly better" without statistical support. With the
seed counts available here, differences are reported as means with dispersion and paired
signs, not as significance claims.

In [ ]:
# --- summary, limitations, reproduction command --------------------------
(RUN_DIR / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2) + "\n")
(RUN_DIR / "git_commits.json").write_text(json.dumps(COMMITS, indent=2) + "\n")
(RUN_DIR / "metrics.json").write_text(json.dumps(METRICS, indent=2, default=str) + "\n")

runtime_report = {
    "run_mode": RUN_MODE, "config_mode": MODE.name,
    "wall_clock_minutes": round((time.time() - STARTED) / 60, 1),
    "validation_stages": VALIDATION,
    "stage_seconds": (RESULT.stage_seconds if RESULT else {}),
    "gpu": GPU_NAME or None,
    "limits": {"max_runtime_hours": MAX_RUNTIME_HOURS,
               "max_temp_disk_gb": MAX_TEMP_DISK_GB, "max_ram_gb": MAX_RAM_GB},
}
(RUN_DIR / "runtime_report.json").write_text(json.dumps(runtime_report, indent=2) + "\n")

reproduce = "\n".join([
    "# Reproduce this run",
    f"git clone --branch {REPO_BRANCH} {REPO_URL} && cd distribution-aware-owod",
    f"git checkout {COMMITS['daowod']}",
    'python -m pip install --editable ".[dev]"',
    f"python experiments/contribution_a.py study --config {CONFIG_PATH} --mode {MODE_FOR_CONFIG} \\",
    f"    --data-root {DATASET_ROOT} --split {SPLIT_FILE} \\",
    (f"    --existing-export {EXISTING_EXPORT} \\" if EXISTING_EXPORT
     else f"    --checkpoint {CHECKPOINT_PATH} \\"),
    f"    --output {RUN_DIR} --cache {CACHE_DIR}",
])
(RUN_DIR / "reproduction_command.txt").write_text(reproduce + "\n")

limitations = f"""# Limitations — {MODE.name}

* **Offline simulation.** No detector is trained or evaluated. known mAP, official
  U-Recall, WI, A-OSE and catastrophic forgetting are NOT AVAILABLE here and are not
  claimed. See metrics.json.
* **Reportability.** mode {MODE.name} is
  {'research-grade' if MODE.research_grade else 'NOT reportable — plumbing validation only'}.
* **Seeds.** {len(MODE.seeds)} seed(s): {list(MODE.seeds)}. Differences are reported as
  means with dispersion and paired signs, not as significance claims.
* **Tail denominators.** Small tail denominators make tail recall move in coarse steps;
  unknown discovery is the primary metric. See the denominator-power warning above.
* **Contribution B.** Allocation core only; not validated experimentally.
* **Representation acquisition.** Incomplete upstream (docs/results.md section 11).
* **LVIS.** Not implemented as a protocol.
"""
(RUN_DIR / "limitations.md").write_text(limitations)

summary = RUN_DIR / "research_summary.md"
if summary.exists():
    display(Markdown(summary.read_text()))
else:
    print("research_summary.md not produced (fixture or interrupted run).")
display(Markdown(limitations))
print("reproduction command ->", RUN_DIR / "reproduction_command.txt")

---
## 15. Artifact export

Builds one compact archive, verifies it by reopening and listing it, records its SHA256
and size, and copies **only** compact artifacts to Drive.

Excluded by default: multi-gigabyte exports, raw representation arrays, dataset images,
checkpoints and build caches. If a large cache must be kept, its external path and digest
are documented rather than copied.

In [ ]:
# --- compact archive + verification --------------------------------------
import zipfile

TIMESTAMP = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
ARCHIVE = Path(OUTPUT_ROOT) / f"daowod_contribution_a_{RUN_MODE.lower()}_{TIMESTAMP}.zip"
INCLUDE_SUFFIXES = {".csv", ".json", ".md", ".txt", ".yaml", ".png"}
INCLUDE_SUFFIXES |= {".pdf"} if SAVE_PDF else set()
MAX_MEMBER_MB = 25

if not CREATE_ZIP:
    print("SKIPPED: CREATE_ZIP = False")
else:
    members, skipped = [], []
    for path in sorted(RUN_DIR.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in INCLUDE_SUFFIXES:
            continue
        if path.stat().st_size > MAX_MEMBER_MB * 1e6:
            skipped.append((path, path.stat().st_size / 1e6)); continue
        members.append(path)

    tmp_archive = ARCHIVE.with_suffix(".partial")
    with zipfile.ZipFile(tmp_archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        for path in members:
            bundle.write(path, path.relative_to(RUN_DIR))
    tmp_archive.rename(ARCHIVE)              # atomic

    with zipfile.ZipFile(ARCHIVE) as bundle:
        listed = bundle.namelist()
        bad = bundle.testzip()
    assert bad is None, f"corrupt member in archive: {bad}"

    expected = ["preflight.csv", "environment.json", "git_commits.json", "metrics.json",
                "limitations.md", "reproduction_command.txt", "runtime_report.json"]
    present = [name for name in expected if name in listed]
    missing = [name for name in expected if name not in listed]

    digest = sha256_of(ARCHIVE)
    print(f"archive : {ARCHIVE.name}")
    print(f"size    : {ARCHIVE.stat().st_size/1e6:.2f} MB   members: {len(listed)}")
    print(f"sha256  : {digest}")
    print(f"contract: {len(present)}/{len(expected)} expected files present")
    if missing:
        print(f"  MISSING (expected for fixture/partial runs): {missing}")
    if skipped:
        print(f"\nexcluded {len(skipped)} oversize file(s) (> {MAX_MEMBER_MB} MB):")
        for path, mb in skipped[:5]:
            print(f"  {mb:8.1f} MB  {path.name}")

    (RUN_DIR / "archive_manifest.json").write_text(json.dumps({
        "archive": ARCHIVE.name, "sha256": digest,
        "size_bytes": ARCHIVE.stat().st_size, "members": len(listed),
        "expected_present": present, "expected_missing": missing,
        "excluded_oversize": [{"name": p.name, "mb": round(mb, 1)} for p, mb in skipped],
        "large_caches_not_copied": {
            "export": EXPORT_REPORT.get("path"),
            "export_sha256": EXPORT_REPORT.get("sha256"),
            "cache_dir": str(CACHE_DIR),
        },
    }, indent=2) + "\n")

    if COPY_COMPACT_RESULTS_TO_DRIVE and USE_DRIVE and Path(DRIVE_ROOT).exists():
        target = Path(DRIVE_ROOT) / "results"
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy2(ARCHIVE, target / ARCHIVE.name)
        print(f"\ncopied compact archive to {target / ARCHIVE.name}")
        print("Large exports and representation arrays were NOT copied; see archive_manifest.json")

---
## 16. Troubleshooting, resume and cleanup

### Resume after a disconnect

Re-run the notebook top to bottom with `RESUME = True` (the default) and the **same**
`RUN_MODE` and `OUTPUT_ROOT`/`CACHE_ROOT`. Completed export chunks and completed stages
are detected and reused, and the log prints what is being reused. Nothing valuable is
deleted automatically.

To force one stage: set `FORCE_STAGE` to `export`, `study`, `audit`, `revealed` or
`representation`.

### Common failures

| Symptom | Cause | Fix |
|---|---|---|
| `daowod imported from …site-packages` | a stale install shadows the clone | Runtime → Restart, then re-run §2 |
| `MultiScaleDeformableAttention` will not import | torch/CUDA mismatch | do **not** upgrade torch; check the version printed in §2 against PROB's requirement |
| `RuntimeBudgetExceeded` | projection exceeds a declared limit | raise `MAX_RUNTIME_HOURS` deliberately, or pick a smaller `RUN_MODE`. The protocol is never auto-shrunk |
| Projected export exceeds disk | `MAX_TEMP_DISK_GB` too low | raise it, lower `MAX_PROPOSALS_PER_IMAGE` (a different experiment — record it), or set `EXISTING_EXPORT` |
| Preflight `FAIL` on split IDs | dataset layout mismatch | check `JPEG_IMAGES_DIR` / `ANNOTATIONS_DIR` against §1 |
| Severity validation stops the run | pool too small to express the axis | use a larger mode; below ~150 evaluation images no axis is expressible |

### Cleanup

The next cell **lists** what it would delete and does nothing unless you set the
confirmation flag. It never touches checkpoints, datasets or frozen exports.

In [ ]:
# --- cleanup: dry run by default -----------------------------------------
CLEAN_TEMPORARY_FILES = False        # set True ONLY after reading the printed list

PROTECTED = ("checkpoint", ".pth", ".pt", "JPEGImages", "Annotations")
candidates = []
for pattern in ("*.partial", ".write_probe"):
    candidates += list(RUN_DIR.rglob(pattern))
candidates += [p for p in Path(CACHE_ROOT).rglob("*.tmp")]
if not KEEP_LARGE_INTERMEDIATES:
    candidates += [p for p in RUN_DIR.rglob("*.npz") if p.stat().st_size > 5e7]

candidates = [p for p in candidates
              if p.is_file() and not any(token in str(p) for token in PROTECTED)]

if not candidates:
    print("Nothing to clean.")
else:
    total = sum(p.stat().st_size for p in candidates) / 1e9
    print(f"PLANNED DELETIONS — {len(candidates)} files, {total:.2f} GB:")
    for path in candidates[:40]:
        print(f"  {path.stat().st_size/1e6:>9.1f} MB  {path}")
    print("\nNever considered:", ", ".join(PROTECTED))
    if CLEAN_TEMPORARY_FILES:
        for path in candidates:
            path.unlink()
        print(f"\nDeleted {len(candidates)} files ({total:.2f} GB).")
    else:
        print("\nDRY RUN. Set CLEAN_TEMPORARY_FILES = True in this cell to delete.")

In [ ]:
# --- final status ---------------------------------------------------------
elapsed = (time.time() - STARTED) / 60
print("=" * 70)
print(f"DAOWOD Contribution A — {RUN_MODE}")
print("=" * 70)
print(f"wall clock   : {elapsed:.1f} min")
print(f"config mode  : {MODE.name} ({'REPORTABLE' if MODE.research_grade else 'NOT reportable'})")
print(f"run dir      : {RUN_DIR}")
print(f"daowod commit: {COMMITS['daowod']}")
print(f"preflight    : {sum(c['status']=='PASS' for c in CHECKS)} pass, "
      f"{sum(c['status']=='WARN' for c in CHECKS)} warn")
print(f"validation   : {sum(v['ok'] for v in VALIDATION.values())}/{len(VALIDATION)} stages")
print()
print("Claimed      : offline discovery metrics over the annotation set")
print("NOT claimed  : known mAP, official U-Recall, WI, A-OSE, forgetting, Contribution B")
print()
print("Next: SMOKE -> DEBUG -> FAST -> MAIN -> MAIN_REVEALED -> REPRESENTATION")